# 🫁 Chest X-Ray Attention Training - All Models

**Training all 7 attention-enhanced models on Kaggle GPU (P100)**

### Models:
1. ResNet50 Baseline
2. ResNet50 + SE
3. ResNet50 + CBAM ⭐
4. ResNet50 + ECA
5. DenseNet121 Baseline
6. DenseNet121 + SE
7. DenseNet121 + CBAM ⭐

**GPU:** Tesla P100 (16GB)  
**Estimated Time:** ~3-4 hours

---

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers scikit-learn matplotlib seaborn tqdm opencv-python-headless pyyaml -q
print('✅ Dependencies installed')

## 2. Verify GPU

In [ ]:
import torch
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🚀 GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Download Dataset

In [ ]:
!kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis -p data/raw --unzip
print('✅ Dataset downloaded')

In [ ]:
from pathlib import Path
data_dir = Path('data/raw')

print(f"\n📁 Dataset Structure:")
for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        n_images = len(list(class_dir.glob('*.png'))) + len(list(class_dir.glob('*.jpg')))
        print(f"  {class_dir.name}: {n_images:,} images")

total = sum(len(list(d.glob('*.png'))) + len(list(d.glob('*.jpg'))) 
           for d in data_dir.iterdir() if d.is_dir())
print(f"\n📊 Total: {total:,} images")

## 4. Create Model Code

In [ ]:
# Create directory structure
Path('src/classifier/models').mkdir(parents=True, exist_ok=True)
Path('src/classifier/utils').mkdir(parents=True, exist_ok=True)
Path('src/classifier/data').mkdir(parents=True, exist_ok=True)
Path('results/models').mkdir(parents=True, exist_ok=True)
print('✅ Directory structure created')

In [ ]:
# Write attention modules
attention_code = '''
import math
import torch
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.channel_fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.channel_fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)
        max_out = self.channel_fc(self.max_pool(x).view(b, c)).view(b, c, 1, 1)
        channel_weight = self.sigmoid(avg_out + max_out)
        x = x * channel_weight
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_input = torch.cat([avg_out, max_out], dim=1)
        spatial_weight = self.sigmoid(self.spatial_conv(spatial_input))
        return x * spatial_weight

class ECA(nn.Module):
    def __init__(self, channels, gamma=2.0, b=1.0):
        super().__init__()
        kernel_size = int(abs((math.log2(channels) + b) / gamma))
        kernel_size = max(3, kernel_size if kernel_size % 2 else kernel_size + 1)
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = x.mean(dim=[2, 3], keepdim=True)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)
'''

with open('src/classifier/models/attention.py', 'w') as f:
    f.write(attention_code)
print('✅ attention.py created')

## 5. Training Configuration

In [ ]:
MODELS = [
    ('resnet50_baseline', 'resnet', 'resnet50', 'none'),
    ('resnet50_se', 'attention', 'resnet50', 'se'),
    ('resnet50_cbam', 'attention', 'resnet50', 'cbam'),
    ('resnet50_eca', 'attention', 'resnet50', 'eca'),
    ('densenet121_baseline', 'densenet', 'densenet121', 'none'),
    ('densenet121_se', 'attention', 'densenet121', 'se'),
    ('densenet121_cbam', 'attention', 'densenet121', 'cbam'),
]

EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
DATA_DIR = 'data/raw'
OUTPUT_DIR = 'results/models'

print(f"📋 Training Configuration:")
print(f"  Models: {len(MODELS)}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")

## 6. Train All Models

In [ ]:
import subprocess
from datetime import datetime

results = []

for i, (model_name, model_type, backbone, attention_type) in enumerate(MODELS, 1):
    print(f"\n{'='*60}")
    print(f"🚀 [{i}/{len(MODELS)}] Training: {model_name}")
    print(f"{'='*60}")
    print(f"  Backbone: {backbone}")
    print(f"  Attention: {attention_type}")
    print(f"  Start: {datetime.now().strftime('%H:%M:%S')}")
    
    cmd = [
        'python', 'train.py',
        '--data_dir', DATA_DIR,
        '--model_type', model_type,
        '--model_name', backbone,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--lr', str(LEARNING_RATE),
        '--output_dir', OUTPUT_DIR,
        '--experiment_name', model_name,
        '--scheduler', 'cosine',
        '--use_amp',
        '--seed', '42'
    ]
    
    if model_type == 'attention' and attention_type != 'none':
        cmd.extend(['--attention_type', attention_type])
    
    try:
        result = subprocess.run(cmd, check=True)
        model_path = Path(OUTPUT_DIR) / model_name / 'best_model.pth'
        if model_path.exists():
            print(f"✅ {model_name} completed!")
            results.append({'model': model_name, 'status': 'success'})
        else:
            print(f"⚠️  {model_name} completed but no checkpoint")
            results.append({'model': model_name, 'status': 'warning'})
    except Exception as e:
        print(f"❌ {model_name} failed: {e}")
        results.append({'model': model_name, 'status': 'failed'})

print(f"\n{'='*60}")
print("📊 TRAINING SUMMARY")
print(f"{'='*60}")
for r in results:
    status = '✅' if r['status'] == 'success' else '⚠️' if r['status'] == 'warning' else '❌'
    print(f"{status} {r['model']}: {r['status']}")
success_count = sum(1 for r in results if r['status'] == 'success')
print(f"\n✅ Successfully trained: {success_count}/{len(MODELS)} models")

## 7. Verify Results

In [ ]:
from pathlib import Path

models_dir = Path(OUTPUT_DIR)
print(f"\n📁 Trained Models:")
print(f"{'='*60}")

for model_dir in sorted(models_dir.iterdir()):
    if model_dir.is_dir():
        checkpoint = model_dir / 'best_model.pth'
        status = '✅' if checkpoint.exists() else '❌'
        size = f"{checkpoint.stat().st_size / 1e6:.1f} MB" if checkpoint.exists() else 'N/A'
        print(f"{status} {model_dir.name:30s} | Size: {size}")

print(f"\n📂 Output: {models_dir.absolute()}")

## 8. Next Steps

### Download Results
```bash
kaggle kernels output luanbhk/chest-x-ray-attention-training-all-7-models -p ./results
```

### Evaluate Locally
```bash
uv run python scripts/evaluate_models.py
uv run python scripts/compare_results.py
uv run python scripts/visualize_results.py
```

---
**Training Complete! 🎉**